In [1]:
import requests
import xml.etree.ElementTree as ET


BUCKET_URL = (
    "https://noaa-gefs-retrospective.s3.amazonaws.com/"
)

PREFIX = (
    "GEFSv12/reforecast/2018/"
    "2018070100/c00/Days:1-10/"
)

response = requests.get(
    BUCKET_URL,
    params={
        "list-type": "2",
        "prefix": PREFIX
    },
    timeout=60
)

response.raise_for_status()

root = ET.fromstring(response.content)

namespace = {
    "s3": "http://s3.amazonaws.com/doc/2006-03-01/"
}

available_files = []

for item in root.findall(
    "s3:Contents",
    namespace
):
    key = item.find(
        "s3:Key",
        namespace
    ).text

    size = int(
        item.find(
            "s3:Size",
            namespace
        ).text
    )

    available_files.append({
        "key": key,
        "filename": key.split("/")[-1],
        "size_mb": size / 1_000_000
    })


print("Files found:", len(available_files))

for file in available_files:
    print(
        file["filename"],
        f"{file['size_mb']:.1f} MB"
    )

Files found: 122
acpcp_sfc_2018070100_c00.grib2 32.1 MB
acpcp_sfc_2018070100_c00.grib2.idx 0.0 MB
apcp_sfc_2018070100_c00.grib2 26.2 MB
apcp_sfc_2018070100_c00.grib2.idx 0.0 MB
cape_sfc_2018070100_c00.grib2 40.7 MB
cape_sfc_2018070100_c00.grib2.idx 0.0 MB
cin_sfc_2018070100_c00.grib2 26.5 MB
cin_sfc_2018070100_c00.grib2.idx 0.0 MB
dlwrf_sfc_2018070100_c00.grib2 39.9 MB
dlwrf_sfc_2018070100_c00.grib2.idx 0.0 MB
dswrf_sfc_2018070100_c00.grib2 27.3 MB
dswrf_sfc_2018070100_c00.grib2.idx 0.0 MB
gflux_sfc_2018070100_c00.grib2 36.6 MB
gflux_sfc_2018070100_c00.grib2.idx 0.0 MB
gust_sfc_2018070100_c00.grib2 43.8 MB
gust_sfc_2018070100_c00.grib2.idx 0.0 MB
hgt_ceiling_2018070100_c00.grib2 83.5 MB
hgt_ceiling_2018070100_c00.grib2.idx 0.0 MB
hgt_hybr_2018070100_c00.grib2 167.7 MB
hgt_hybr_2018070100_c00.grib2.idx 0.0 MB
hgt_pres_2018070100_c00.grib2 476.2 MB
hgt_pres_2018070100_c00.grib2.idx 0.0 MB
hgt_pres_abv700mb_2018070100_c00.grib2 283.3 MB
hgt_pres_abv700mb_2018070100_c00.grib2.idx 0.1 MB
hg

In [2]:
search_terms = [
    "pres",
    "prmsl",
    "ugrd",
    "vgrd",
    "rh",
    "tmp",
    "hgt"
]


predictor_files = []

for file in available_files:
    filename = file["filename"].lower()

    if any(
        term in filename
        for term in search_terms
    ):
        predictor_files.append(file)


print(
    "Potential predictor files:",
    len(predictor_files)
)

for index, file in enumerate(predictor_files):
    print(
        f"[{index}]",
        file["filename"],
        f"{file['size_mb']:.1f} MB"
    )

Potential predictor files: 66
[0] hgt_ceiling_2018070100_c00.grib2 83.5 MB
[1] hgt_ceiling_2018070100_c00.grib2.idx 0.0 MB
[2] hgt_hybr_2018070100_c00.grib2 167.7 MB
[3] hgt_hybr_2018070100_c00.grib2.idx 0.0 MB
[4] hgt_pres_2018070100_c00.grib2 476.2 MB
[5] hgt_pres_2018070100_c00.grib2.idx 0.0 MB
[6] hgt_pres_abv700mb_2018070100_c00.grib2 283.3 MB
[7] hgt_pres_abv700mb_2018070100_c00.grib2.idx 0.1 MB
[8] hgt_sfc_2018070100_c00.grib2 36.2 MB
[9] hgt_sfc_2018070100_c00.grib2.idx 0.0 MB
[10] hlcy_hgt_2018070100_c00.grib2 47.6 MB
[11] hlcy_hgt_2018070100_c00.grib2.idx 0.0 MB
[12] pbl_hgt_2018070100_c00.grib2 101.6 MB
[13] pbl_hgt_2018070100_c00.grib2.idx 0.0 MB
[14] pres_hybr_2018070100_c00.grib2 266.2 MB
[15] pres_hybr_2018070100_c00.grib2.idx 0.0 MB
[16] pres_msl_2018070100_c00.grib2 73.2 MB
[17] pres_msl_2018070100_c00.grib2.idx 0.0 MB
[18] pres_mslet_2018070100_c00.grib2 71.7 MB
[19] pres_mslet_2018070100_c00.grib2.idx 0.0 MB
[20] pres_pvor_2018070100_c00.grib2 62.4 MB
[21] pres_pvor_

In [3]:
INITIALIZATION = "2018070100"

INDEX_FILES = {
    "mslp": (
        f"pres_msl_{INITIALIZATION}_c00.grib2.idx"
    ),
    "u850": (
        f"ugrd_pres_{INITIALIZATION}_c00.grib2.idx"
    ),
    "v850": (
        f"vgrd_pres_{INITIALIZATION}_c00.grib2.idx"
    ),
    "q850": (
        f"spfh_pres_{INITIALIZATION}_c00.grib2.idx"
    )
}


index_contents = {}


for predictor, filename in INDEX_FILES.items():
    url = (
        BUCKET_URL
        + PREFIX
        + filename
    )

    response = requests.get(
        url,
        timeout=60
    )

    response.raise_for_status()

    lines = [
        line
        for line in response.text.splitlines()
        if line.strip()
    ]

    index_contents[predictor] = lines

    print(
        predictor,
        "| index entries:",
        len(lines)
    )

mslp | index entries: 80
u850 | index entries: 560
v850 | index entries: 560
q850 | index entries: 560


In [4]:
mslp_lines = index_contents["mslp"]

for line in mslp_lines[:30]:
    print(line)

1:0:d=2018070100:PRES:mean sea level:3 hour fcst:ENS=low-res ctl
2:891622:d=2018070100:PRES:mean sea level:6 hour fcst:ENS=low-res ctl
3:1798784:d=2018070100:PRES:mean sea level:9 hour fcst:ENS=low-res ctl
4:2713877:d=2018070100:PRES:mean sea level:12 hour fcst:ENS=low-res ctl
5:3634498:d=2018070100:PRES:mean sea level:15 hour fcst:ENS=low-res ctl
6:4556955:d=2018070100:PRES:mean sea level:18 hour fcst:ENS=low-res ctl
7:5483095:d=2018070100:PRES:mean sea level:21 hour fcst:ENS=low-res ctl
8:6409034:d=2018070100:PRES:mean sea level:24 hour fcst:ENS=low-res ctl
9:7334039:d=2018070100:PRES:mean sea level:27 hour fcst:ENS=low-res ctl
10:8222151:d=2018070100:PRES:mean sea level:30 hour fcst:ENS=low-res ctl
11:9111050:d=2018070100:PRES:mean sea level:33 hour fcst:ENS=low-res ctl
12:10003376:d=2018070100:PRES:mean sea level:36 hour fcst:ENS=low-res ctl
13:10903274:d=2018070100:PRES:mean sea level:39 hour fcst:ENS=low-res ctl
14:11803075:d=2018070100:PRES:mean sea level:42 hour fcst:ENS=low-re

In [5]:
for predictor in [
    "u850",
    "v850",
    "q850"
]:
    matches = [
        line
        for line in index_contents[predictor]
        if "850 mb" in line
    ]

    print(
        f"\n{predictor} 850-hPa entries:",
        len(matches)
    )

    for line in matches[:30]:
        print(line)


u850 850-hPa entries: 80
6:4061199:d=2018070100:UGRD:850 mb:3 hour fcst:ENS=low-res ctl
13:9699684:d=2018070100:UGRD:850 mb:6 hour fcst:ENS=low-res ctl
20:15368489:d=2018070100:UGRD:850 mb:9 hour fcst:ENS=low-res ctl
27:21108658:d=2018070100:UGRD:850 mb:12 hour fcst:ENS=low-res ctl
34:26894186:d=2018070100:UGRD:850 mb:15 hour fcst:ENS=low-res ctl
41:32753008:d=2018070100:UGRD:850 mb:18 hour fcst:ENS=low-res ctl
48:38625083:d=2018070100:UGRD:850 mb:21 hour fcst:ENS=low-res ctl
55:44498945:d=2018070100:UGRD:850 mb:24 hour fcst:ENS=low-res ctl
62:50364482:d=2018070100:UGRD:850 mb:27 hour fcst:ENS=low-res ctl
69:56219231:d=2018070100:UGRD:850 mb:30 hour fcst:ENS=low-res ctl
76:62086875:d=2018070100:UGRD:850 mb:33 hour fcst:ENS=low-res ctl
83:67989894:d=2018070100:UGRD:850 mb:36 hour fcst:ENS=low-res ctl
90:73919280:d=2018070100:UGRD:850 mb:39 hour fcst:ENS=low-res ctl
97:79877646:d=2018070100:UGRD:850 mb:42 hour fcst:ENS=low-res ctl
104:85866790:d=2018070100:UGRD:850 mb:45 hour fcst:ENS=l

In [6]:
selected_index_lines = {}


for predictor, lines in index_contents.items():
    if predictor == "mslp":
        level_matches = lines
    else:
        level_matches = [
            line
            for line in lines
            if "850 mb" in line
        ]

    hour_matches = [
        line
        for line in level_matches
        if (
            "36 hour" in line.lower()
            or "36-hour" in line.lower()
            or ":36:" in line.lower()
        )
    ]

    selected_index_lines[predictor] = hour_matches

    print(f"\n{predictor} +36-hour matches:")

    for line in hour_matches:
        print(line)


mslp +36-hour matches:
12:10003376:d=2018070100:PRES:mean sea level:36 hour fcst:ENS=low-res ctl

u850 +36-hour matches:
83:67989894:d=2018070100:UGRD:850 mb:36 hour fcst:ENS=low-res ctl

v850 +36-hour matches:
83:68072719:d=2018070100:VGRD:850 mb:36 hour fcst:ENS=low-res ctl

q850 +36-hour matches:
83:53966812:d=2018070100:SPFH:850 mb:36 hour fcst:ENS=low-res ctl


In [7]:
from pathlib import Path
import time

import pandas as pd
import requests
import xarray as xr


PROJECT = Path(r"Z:\Projects\monsoon-postprocessing")

PREDICTOR_FOLDER = (
    PROJECT
    / "data"
    / "raw"
    / "gefs_predictors"
)

PREDICTOR_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


PREDICTOR_SPECS = {
    "mslp": {
        "file_prefix": "pres_msl",
        "search_text": (
            ":PRES:mean sea level:36 hour fcst:"
        )
    },
    "u850": {
        "file_prefix": "ugrd_pres",
        "search_text": (
            ":UGRD:850 mb:36 hour fcst:"
        )
    },
    "v850": {
        "file_prefix": "vgrd_pres",
        "search_text": (
            ":VGRD:850 mb:36 hour fcst:"
        )
    },
    "q850": {
        "file_prefix": "spfh_pres",
        "search_text": (
            ":SPFH:850 mb:36 hour fcst:"
        )
    }
}

In [8]:
def download_predictor_message(
    initialization,
    predictor_name
):
    """Download one GRIB message using its index offsets."""

    spec = PREDICTOR_SPECS[predictor_name]

    year = initialization[:4]

    directory = (
        f"GEFSv12/reforecast/{year}/"
        f"{initialization}/c00/Days:1-10/"
    )

    base_name = (
        f"{spec['file_prefix']}_"
        f"{initialization}_c00.grib2"
    )

    grib_url = (
        BUCKET_URL
        + directory
        + base_name
    )

    index_url = grib_url + ".idx"

    destination = (
        PREDICTOR_FOLDER
        / (
            f"{predictor_name}_"
            f"{initialization}_f036.grib2"
        )
    )

    if destination.exists() and destination.stat().st_size > 0:
        return destination

    index_response = requests.get(
        index_url,
        timeout=60
    )

    index_response.raise_for_status()

    lines = [
        line
        for line in index_response.text.splitlines()
        if line.strip()
    ]

    matching_positions = [
        position
        for position, line in enumerate(lines)
        if spec["search_text"] in line
    ]

    if len(matching_positions) != 1:
        raise ValueError(
            f"Expected one {predictor_name} match for "
            f"{initialization}, found "
            f"{len(matching_positions)}"
        )

    position = matching_positions[0]

    start_byte = int(
        lines[position].split(":")[1]
    )

    if position + 1 < len(lines):
        end_byte = (
            int(
                lines[position + 1].split(":")[1]
            )
            - 1
        )
    else:
        head_response = requests.head(
            grib_url,
            timeout=60
        )

        head_response.raise_for_status()

        end_byte = (
            int(
                head_response.headers[
                    "content-length"
                ]
            )
            - 1
        )

    range_response = requests.get(
        grib_url,
        headers={
            "Range": (
                f"bytes={start_byte}-{end_byte}"
            )
        },
        timeout=(30, 180)
    )

    # Status 206 confirms that only the requested range was returned.
    if range_response.status_code != 206:
        raise RuntimeError(
            f"Server returned status "
            f"{range_response.status_code} "
            f"instead of 206."
        )

    expected_size = (
        end_byte - start_byte + 1
    )

    if len(range_response.content) != expected_size:
        raise IOError(
            "Downloaded byte count does not match "
            "the requested range."
        )

    if not range_response.content.startswith(b"GRIB"):
        raise IOError(
            "Downloaded message does not begin with GRIB."
        )

    temporary_file = destination.with_suffix(
        ".tmp.grib2"
    )

    with open(temporary_file, "wb") as output:
        output.write(range_response.content)

    temporary_file.replace(destination)

    return destination

In [9]:
TEST_INITIALIZATION = "2018070100"

test_files = []


for predictor_name in PREDICTOR_SPECS:
    downloaded_file = download_predictor_message(
        TEST_INITIALIZATION,
        predictor_name
    )

    test_files.append(downloaded_file)

    print(
        predictor_name,
        "|",
        downloaded_file.name,
        "|",
        f"{downloaded_file.stat().st_size / 1_000_000:.2f} MB"
    )

mslp | mslp_2018070100_f036.grib2 | 0.90 MB
u850 | u850_2018070100_f036.grib2 | 0.84 MB
v850 | v850_2018070100_f036.grib2 | 0.86 MB
q850 | q850_2018070100_f036.grib2 | 0.70 MB


In [10]:
for file in test_files:
    print("\nFile:", file.name)

    dataset = xr.open_dataset(
        file,
        engine="cfgrib",
        backend_kwargs={
            "indexpath": ""
        }
    )

    print("Variables:", list(dataset.data_vars))
    print("Dimensions:", dict(dataset.sizes))
    print("Coordinates:", list(dataset.coords))

    for variable in dataset.data_vars:
        print(
            variable,
            "| units:",
            dataset[variable].attrs.get("units")
        )

    dataset.close()


File: mslp_2018070100_f036.grib2
Variables: ['msl']
Dimensions: {'latitude': 721, 'longitude': 1440}
Coordinates: ['number', 'time', 'step', 'meanSea', 'latitude', 'longitude', 'valid_time']
msl | units: Pa

File: u850_2018070100_f036.grib2
Variables: ['u']
Dimensions: {'latitude': 721, 'longitude': 1440}
Coordinates: ['number', 'time', 'step', 'isobaricInhPa', 'latitude', 'longitude', 'valid_time']
u | units: m s**-1

File: v850_2018070100_f036.grib2
Variables: ['v']
Dimensions: {'latitude': 721, 'longitude': 1440}
Coordinates: ['number', 'time', 'step', 'isobaricInhPa', 'latitude', 'longitude', 'valid_time']
v | units: m s**-1

File: q850_2018070100_f036.grib2
Variables: ['q']
Dimensions: {'latitude': 721, 'longitude': 1440}
Coordinates: ['number', 'time', 'step', 'isobaricInhPa', 'latitude', 'longitude', 'valid_time']
q | units: kg kg**-1


In [11]:
initialization_dates = pd.date_range(
    start="2018-06-30",
    end="2018-07-30",
    freq="D"
)

download_rows = []


for date in initialization_dates:
    initialization = date.strftime(
        "%Y%m%d00"
    )

    for predictor_name in PREDICTOR_SPECS:
        try:
            file = download_predictor_message(
                initialization,
                predictor_name
            )

            download_rows.append({
                "initialization": initialization,
                "predictor": predictor_name,
                "success": True,
                "size_mb": (
                    file.stat().st_size
                    / 1_000_000
                )
            })

            print(
                "Downloaded:",
                initialization,
                predictor_name
            )

        except Exception as error:
            download_rows.append({
                "initialization": initialization,
                "predictor": predictor_name,
                "success": False,
                "size_mb": np.nan
            })

            print(
                "Failed:",
                initialization,
                predictor_name,
                "|",
                error
            )

    time.sleep(0.1)

Downloaded: 2018063000 mslp
Downloaded: 2018063000 u850
Downloaded: 2018063000 v850
Downloaded: 2018063000 q850
Downloaded: 2018070100 mslp
Downloaded: 2018070100 u850
Downloaded: 2018070100 v850
Downloaded: 2018070100 q850
Downloaded: 2018070200 mslp
Downloaded: 2018070200 u850
Downloaded: 2018070200 v850
Downloaded: 2018070200 q850
Downloaded: 2018070300 mslp
Downloaded: 2018070300 u850
Downloaded: 2018070300 v850
Downloaded: 2018070300 q850
Downloaded: 2018070400 mslp
Downloaded: 2018070400 u850
Downloaded: 2018070400 v850
Downloaded: 2018070400 q850
Downloaded: 2018070500 mslp
Downloaded: 2018070500 u850
Downloaded: 2018070500 v850
Downloaded: 2018070500 q850
Downloaded: 2018070600 mslp
Downloaded: 2018070600 u850
Downloaded: 2018070600 v850
Downloaded: 2018070600 q850
Downloaded: 2018070700 mslp
Downloaded: 2018070700 u850
Downloaded: 2018070700 v850
Downloaded: 2018070700 q850
Downloaded: 2018070800 mslp
Downloaded: 2018070800 u850
Downloaded: 2018070800 v850
Downloaded: 20180708

In [12]:
download_df = pd.DataFrame(
    download_rows
)

print(
    "Successful files:",
    int(download_df["success"].sum())
)

print(
    "Failed files:",
    int((~download_df["success"]).sum())
)

print(
    "Total size:",
    download_df["size_mb"].sum(),
    "MB"
)

display(
    download_df[
        ~download_df["success"]
    ]
)

Successful files: 124
Failed files: 0
Total size: 100.231296 MB


,initialization,predictor,success,size_mb
